<a href="https://colab.research.google.com/github/lulusbersamasama-TE/solar-ai-mppt-optimization/blob/main/notebooks/02_gmpp_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 — GMPP voltage regression under partial shading

Each observation is a **complete I–V curve**, not an individual voltage sweep point. Predict the global power peak voltage from six section irradiances. Compare linear regression to random forest. All model features must be available at inference; using six section irradiances is an explicit sensor assumption.

In [ ]:
from google.colab import drive
from pathlib import Path
import os

# Menghubungkan Google Drive
drive.mount('/content/drive')

# Folder proyek di Google Drive
project = Path(
    '/content/drive/MyDrive/solar_ai_mppt_project'
)

# Memastikan folder tersedia
(project / 'data').mkdir(parents=True, exist_ok=True)
(project / 'outputs').mkdir(parents=True, exist_ok=True)

# Menyesuaikan lokasi dengan notebook
link = Path('/content/solar_ai_mppt_project')

if not link.exists():
    os.symlink(
        project,
        link,
        target_is_directory=True
    )

# Memvalidasi file dataset
required = [
    'srrl_weather_compact.csv.gz',
    'pv_database_summary_extract.csv',
    'pv_iv_curves_0p1V.npz'
]

for filename in required:
    path = project / 'data' / filename
    print(filename, 'OK' if path.exists() else 'MISSING')


In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
ROOT=next((p for p in [Path.cwd(),Path.cwd().parent,Path('/content/solar_ai_mppt_project')]
           if (p/'data'/'pv_database_summary_extract.csv').exists()),None)
if ROOT is None: raise FileNotFoundError('Read data/README.md and place the prepared data in data/')
DATA=ROOT/'data';OUT=ROOT/'outputs';OUT.mkdir(exist_ok=True)
FAST=os.getenv('SOLAR_FAST','0')=='1'; SEED=42

## A. Inspect electrical labels and possible distribution shift

In [ ]:
pv=pd.read_csv(DATA/'pv_database_summary_extract.csv')
FEATURES=[f'Section_{j}_Wm2' for j in range(1,7)]
pv['curve_id']=pv.Curve.astype(int)
assert pv.curve_id.is_unique and len(pv)==14400
assert set(FEATURES+['Vmp_V','Imp_A','Pmax_W']).issubset(pv.columns)
err=(pv.Vmp_V*pv.Imp_A-pv.Pmax_W).abs()
print('Curves',len(pv),'max label power inconsistency',err.max())
print('Ranges Vmp (V)',pv.Vmp_V.min(),pv.Vmp_V.max(),
      'Pmax (W)',pv.Pmax_W.min(),pv.Pmax_W.max())
print('Uniform section patterns',int(pv[FEATURES].nunique(axis=1).eq(1).sum()))
print(pv[FEATURES+['Vmp_V','Pmax_W']].head())
assert err.max()<0.005

## B. Group-disjoint 70/15/15 split

Groups share the first five substring irradiances. This prevents distinct sixth-section variants of identical first-five patterns being split across train/validation/test. Not an independent-plant validation.

In [ ]:
groups=pv[FEATURES[:5]].astype(int).astype(str).agg('|'.join,axis=1)
outer=GroupShuffleSplit(n_splits=1,test_size=.15,random_state=SEED)
dev_idx,test_idx=next(outer.split(pv,groups=groups))
inner=GroupShuffleSplit(n_splits=1,test_size=.15/.85,random_state=SEED+1)
tr_local,va_local=next(inner.split(pv.iloc[dev_idx],groups=groups.iloc[dev_idx]))
tr_idx=dev_idx[tr_local];va_idx=dev_idx[va_local]
split=np.full(len(pv),'unused',dtype=object)
split[tr_idx]='train';split[va_idx]='validation';split[test_idx]='test'
pv['split']=split
assert 'unused' not in set(split)
for a,b in [('train','test'),('validation','test'),('train','validation')]:
    assert not (set(groups[pv.split==a]) & set(groups[pv.split==b]))
print(pv.split.value_counts())
pv[['curve_id','split']].to_csv(OUT/'gmpp_splits.csv',index=False)

## C. Train, evaluate, export

Never include Vmp or Pmax labels as model inputs.

In [ ]:
train=pv[pv.split=='train']; valid=pv[pv.split=='validation'];test=pv[pv.split=='test']
linear=LinearRegression().fit(train[FEATURES],train.Vmp_V)
rf=RandomForestRegressor(n_estimators=50 if FAST else 180,max_depth=20,
                         min_samples_leaf=2,n_jobs=-1,random_state=SEED)
rf.fit(train[FEATURES],train.Vmp_V)
def pred(model,d): return np.clip(model.predict(d[FEATURES]),0,41.4)
rows=[]
for name,d in [('validation',valid),('test',test)]:
    for model_name,model in [('Linear Regression',linear),('Random Forest',rf)]:
        yhat=pred(model,d)
        rows.append({'subset':name,'model':model_name,
            'MAE_V':mean_absolute_error(d.Vmp_V,yhat),
            'RMSE_V':np.sqrt(mean_squared_error(d.Vmp_V,yhat)),
            'R2':r2_score(d.Vmp_V,yhat),
            'within_1V_pct':100*np.mean(np.abs(yhat-d.Vmp_V)<=1)})
metrics=pd.DataFrame(rows)
print(metrics.round(3).to_string(index=False))
metrics.to_csv(OUT/'gmpp_regression_metrics.csv',index=False)
pd.DataFrame({'curve_id':test.curve_id,'Vmp_V':test.Vmp_V,
    'v_pred_linear':pred(linear,test),'v_pred_rf':pred(rf,test)}
    ).to_csv(OUT/'gmpp_test_predictions.csv',index=False)
joblib.dump({'linear':linear,'random_forest':rf,'feature_columns':FEATURES},
    OUT/'gmpp_models.joblib',compress=3)

## D. Diagnostics: voltage errors and permutation importance

Feature importance measures predictive contribution on validation only, not physical causality.

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].scatter(test.Vmp_V,pred(linear,test),s=7,alpha=.35,label='Linear')
axes[0].scatter(test.Vmp_V,pred(rf,test),s=7,alpha=.35,label='RF')
axes[0].plot([0,42],[0,42],color='black',ls=':')
axes[0].set(xlabel='V GMPP actual (V)',ylabel='Predicted V (V)',title='Held-out test')
axes[0].legend()
axes[1].hist(pred(rf,test)-test.Vmp_V,bins=60)
axes[1].set(xlabel='RF voltage error (V)',ylabel='Count')
plt.tight_layout();fig.savefig(OUT/'gmpp_voltage_predictions.png',dpi=150);plt.show()
vi=valid.sample(n=min(600 if FAST else 1000,len(valid)),random_state=SEED)
imp=permutation_importance(rf,vi[FEATURES],vi.Vmp_V,n_repeats=3,
                           random_state=SEED,n_jobs=1,scoring='neg_mean_absolute_error')
order=np.argsort(imp.importances_mean)[::-1]
fig,ax=plt.subplots(figsize=(9,4))
ax.bar(np.array(FEATURES)[order],imp.importances_mean[order])
ax.set(ylabel='MAE increase when shuffled (V)',title='Validation permutation importance')
plt.xticks(rotation=30);plt.tight_layout()
fig.savefig(OUT/'gmpp_permutation_importance.png',dpi=150);plt.show()
print('Saved outputs:',OUT)

## Interpretation

GMPP voltage RMSE cannot be interpreted as harvested-power efficiency. Evaluate each proposed voltage on the held-out *measured/simulated source* P–V curves in Notebook 03. A stronger sensor-parity comparison with P&O remains future work.